# LAB 07 — Lakeflow Spark Declarative Pipelines

> **Ticket DATA-221** · From: Head of Data Platform · Priority: High
>
> One declarative pipeline must build the RetailHub sales star: Bronze from the landing files, Silver with data-quality rules, the dimensions `dim_customer` and `dim_product`, and the fact table `fact_sales`. Bad orders must not reach the facts, but they must not disappear either: keep them where someone can fix and replay them. Marketing needs the history of customer changes; operations need only the current state of each customer.

**Definition of done** (the tasks of Lab 07 in the Workshop Guide app)
- Start: the pipeline writes to the schema `sdp_lab` in your catalog and reads the landing folders that this notebook creates (`source_path`).
- Task 1: the three Bronze streaming tables hold every row of the landed files, with the file path and the arrival time of each row.
- Task 2: `silver_orders` has no order without an ID, a customer or an order time. It keeps the returns (negative amounts), and the event log counts all four rules.
- Task 3: every Bronze order is in `silver_orders` or in `silver_orders_quarantine`, never in both.
- Task 4: `silver_customers` has one current row per customer and no history.
- Task 5: `dim_customer` keeps every version of a customer, with `__START_AT` and `__END_AT`.
- Task 6: `dim_product` has one row per product.
- Task 7: `fact_sales` has one row for each Silver order, joined with the current customer and the product.
- Task 8 (stretch): a rule that stops the update fails on the returns, and then you put back the rule of Task 2.
- Task 9 (stretch): `daily_sales` adds up to `fact_sales`.

You build the pipeline in the **Lakeflow Pipelines Editor**, in the skeleton folder of your language: `notebooks/day2/lab/lab_07_pipeline/sql` or `.../python`. The Workshop Guide gives you the code of each task. This notebook lands the source files and checks the pipeline after each run.

## Setup

In [ ]:
%run ../../setup/00_setup

In [ ]:
# Lab 07 names: the pipeline schema and the landing folders it reads (source_path)
import os

PIPELINE_SCHEMA = "sdp_lab"
SOURCE_PATH = f"{DATASET_PATH}/sdp_lab"


def T(name):
    """Full name of a pipeline table."""
    return f"{CATALOG}.{PIPELINE_SCHEMA}.{name}"


def n(query):
    """First value of a SQL query."""
    return spark.sql(query).first()[0]


print(f"Catalog: {CATALOG}")
print(f"Schema:  {PIPELINE_SCHEMA}")

### Wave 1: the first files

The pipeline reads three landing folders under `source_path`: `orders`, `customers` and `products`. Run this cell once, before the first pipeline run, so that the pipeline has files to read. It copies the order batch file and writes the first customer file and the first product file. It also prints the value of `source_path` for the pipeline configuration.

A real source system sends one row per customer and per product in each file. The course files have repeated IDs and rows without an ID, so the cell keeps one row per ID. The cell skips a file that is already in the landing folder, so a second run changes nothing until you delete the folder.

In [ ]:
# Wave 1: land the first order, customer and product files
def land_one_row_per_key(source, key, order_by, target):
    """Write one row per key of a CSV file to a landing file; skip the file when it is already there."""
    if os.path.exists(target):
        return f"{os.path.basename(target)}: already landed"
    df = spark.sql(f"""
        SELECT * EXCEPT (_rn, _rescued_data) FROM (
          SELECT *, row_number() OVER (PARTITION BY {key} ORDER BY {order_by}) AS _rn
          FROM read_files('{source}', format => 'csv', header => true)
          WHERE {key} IS NOT NULL)
        WHERE _rn = 1""")
    pdf = df.toPandas()
    pdf.to_csv(target, index=False)
    return f"{os.path.basename(target)}: {len(pdf):,} rows"


for folder in ("orders", "customers", "products"):
    dbutils.fs.mkdirs(f"{SOURCE_PATH}/{folder}")

orders_file = f"{SOURCE_PATH}/orders/orders_batch.json"
if os.path.exists(orders_file):
    print("orders_batch.json: already landed")
else:
    dbutils.fs.cp(f"{DATASET_PATH}/orders/orders_batch.json", orders_file)
    print("orders_batch.json: landed")
print(land_one_row_per_key(f"{DATASET_PATH}/customers/customers.csv", "customer_id", "registration_date DESC, email", f"{SOURCE_PATH}/customers/customers_000_initial.csv"))
print(land_one_row_per_key(f"{DATASET_PATH}/products/products.csv", "product_id", "product_name, list_price DESC", f"{SOURCE_PATH}/products/products_000_initial.csv"))

print("\nValue for the pipeline configuration:")
print(f"  source_path = {SOURCE_PATH}")

## Build the pipeline

Now go to the Workshop Guide. In **Start** you create the pipeline from the folder of your language. Tasks 1–3 add Bronze, Silver and the quarantine, and at the end of Task 3 you run the pipeline for the first time. After each pipeline run, come back to this notebook and run the check cell of that run.

> **`NameError` in a check cell** (for example `name 'n' is not defined` or `name 'T' is not defined`): after a long break the notebook session has expired and lost the names of Setup. Run the two Setup cells again (`%run` and **Lab 07 names**). For the Run 3 check, also run the Wave 2 cell again: it skips the file that has already landed, so nothing changes.

## Run 1: check

Run this cell after Tasks 1–3 and the first pipeline run. It checks that no order gets lost between the layers: Bronze holds every landed row, and each Bronze order is in Silver or in the quarantine. It also checks that the returns stay in Silver. The last output shows the four rules of `silver_orders` from the event log, with the same numbers as the data quality of `silver_orders` in the editor. If the cell fails with `NameError`, see the note under **Build the pipeline**.

In [ ]:
# Run 1: check Bronze, Silver and the quarantine, and show the four rules from the event log
GOOD = "order_id IS NOT NULL AND customer_id IS NOT NULL AND order_datetime IS NOT NULL"
OPTIONS = {"orders": "format => 'json'", "customers": "format => 'csv', header => true", "products": "format => 'csv', header => true"}
for folder, options in OPTIONS.items():
    bronze = spark.table(T(f"bronze_{folder}"))
    assert {"_source_file", "_ingested_at"} <= set(bronze.columns), f"bronze_{folder} needs _source_file and _ingested_at (Task 1)"
    files = [r[0] for r in bronze.select("_source_file").distinct().collect()]
    assert files, f"bronze_{folder} is empty (Task 1)"
    for f in files:  # every file that Bronze read is there in full
        rows = n(f"SELECT count(*) FROM read_files('{f}', {options})")
        assert bronze.where(bronze._source_file == f).count() == rows, f"bronze_{folder} does not have all {rows:,} rows of {f} (Task 1)"

bronze = spark.table(T("bronze_orders")).count()
silver = spark.table(T("silver_orders")).count()
quarantine = spark.table(T("silver_orders_quarantine")).count()
bad_in_silver = spark.table(T("silver_orders")).where("order_id IS NULL OR customer_id IS NULL OR order_ts IS NULL").count()
good_in_quarantine = spark.table(T("silver_orders_quarantine")).where(GOOD).count()
returns_bronze = spark.table(T("bronze_orders")).where(f"{GOOD} AND total_amount < 0").count()
returns_silver = spark.table(T("silver_orders")).where("total_amount < 0").count()

assert bad_in_silver == 0, f"silver_orders has {bad_in_silver:,} orders that a drop rule must remove (Task 2)"
assert returns_silver == returns_bronze, f"silver_orders has {returns_silver:,} returns, Bronze {returns_bronze:,}: the amount rule must only count (Task 2)"
assert good_in_quarantine == 0, f"The quarantine has {good_in_quarantine:,} good orders (Task 3)"
assert silver + quarantine == bronze, f"silver {silver:,} + quarantine {quarantine:,} != bronze {bronze:,} (Task 3)"
print(f"Run 1 OK: bronze {bronze:,} = silver {silver:,} + quarantine {quarantine:,}; {returns_silver:,} returns kept in Silver")

display(spark.sql(f"""
    SELECT exp.name AS rule, sum(exp.passed_records) AS passed, sum(exp.failed_records) AS failed
    FROM (
      SELECT explode(from_json(details:flow_progress.data_quality.expectations,
        'array<struct<name:string, dataset:string, passed_records:bigint, failed_records:bigint>>')) AS exp
      FROM event_log(TABLE({T('silver_orders')}))
      WHERE event_type = 'flow_progress'
        AND details:flow_progress.data_quality.expectations IS NOT NULL)
    WHERE endswith(exp.dataset, 'silver_orders')
    GROUP BY exp.name ORDER BY exp.name"""))

## Run 2: check

Run this cell after Tasks 4–5 and the second pipeline run. Both customer tables read the same Bronze table. `silver_customers` keeps only the current row of each customer, and `dim_customer` adds the columns `__START_AT` and `__END_AT`. Only one customer file has landed so far, so every customer has one version, and it is the current one. If the cell fails with `NameError`, see the note under **Build the pipeline**.

In [ ]:
# Run 2: check the current customers and the history columns
customers = n(f"SELECT count(DISTINCT customer_id) FROM {T('bronze_customers')}")
assert "__START_AT" not in spark.table(T("silver_customers")).columns, "silver_customers must not keep history (Task 4)"
assert {"__START_AT", "__END_AT"} <= set(spark.table(T("dim_customer")).columns), "dim_customer needs __START_AT and __END_AT (Task 5)"
rows, ids = spark.sql(f"SELECT count(*), count(DISTINCT customer_id) FROM {T('silver_customers')}").first()
assert rows == ids == customers, f"silver_customers needs one row for each of the {customers:,} customers (Task 4)"
rows, ids = spark.sql(f"SELECT count(*), count(DISTINCT customer_id) FROM {T('dim_customer')} WHERE __END_AT IS NULL").first()
assert rows == ids == customers, f"dim_customer needs one current row for each of the {customers:,} customers (Task 5)"
print(f"Run 2 OK: {customers:,} customers in silver_customers (current state) and dim_customer (with history columns)")

## Wave 2: customer changes

Customer data changes over time, and the source system sends the changes in a new file. This change file has existing customers with at least one changed column and some new customers. The cell lands the file and shows, for each customer in it, whether the customer is changed, new or unchanged. The table shows only city and segment, so a changed customer can look the same there, because another column changed (for example the e-mail or the name).

Run this cell after the check of Run 2. Then do Tasks 6 and 7 before the third pipeline run. The change file arrives later than the first file, so AUTO CDC applies its rows as the newer versions.

In [ ]:
# Wave 2: land the change file and show which customers change and which are new
changes_file = f"{SOURCE_PATH}/customers/customers_001_changes.csv"
already_landed = os.path.exists(changes_file)
if not already_landed:
    dbutils.fs.cp(f"{DATASET_PATH}/customers/customers_new.csv", changes_file)

FIRST = f"read_files('{SOURCE_PATH}/customers/customers_000_initial.csv', format => 'csv', header => true)"
CHANGES = f"read_files('{changes_file}', format => 'csv', header => true)"
# The comparison as a query text: the Run 3 check counts the changed customers with it
WAVE2 = f"""
    SELECT c.customer_id,
           CASE WHEN f.customer_id IS NULL THEN 'new'
                WHEN to_json(struct(c.* EXCEPT (_rescued_data))) = to_json(struct(f.* EXCEPT (_rescued_data))) THEN 'unchanged'
                ELSE 'changed' END AS change,
           f.city AS city_before, c.city AS city_after,
           f.customer_segment AS segment_before, c.customer_segment AS segment_after
    FROM {CHANGES} c LEFT JOIN {FIRST} f USING (customer_id)"""
display(spark.sql(f"{WAVE2} ORDER BY change, customer_id"))
counts = {r["change"]: r["n"] for r in spark.sql(f"SELECT change, count(*) AS n FROM ({WAVE2}) GROUP BY change").collect()}
summary = f"{counts.get('changed', 0)} changed, {counts.get('new', 0)} new, {counts.get('unchanged', 0)} unchanged customers"
if already_landed:
    print(f"customers_001_changes.csv: already landed ({summary} in the file)")
else:
    print(f"Wave 2 landed: {summary}")

## Run 3: check

Run this cell after Tasks 6–7 and the third pipeline run. A changed customer now has two versions in `dim_customer` and one row in `silver_customers`. `fact_sales` has one row for each Silver order. An order with an unknown customer or product stays in the facts, with empty customer or product columns. The last output shows the two versions of `CUST000001` next to its current row in `silver_customers`. If the cell fails with `NameError`, see the note under **Build the pipeline**.

In [ ]:
# Run 3: check the customer history, the product dimension and the facts
changed = n(f"SELECT count(*) FROM ({WAVE2}) WHERE change = 'changed'")
customers = n(f"SELECT count(DISTINCT customer_id) FROM {T('bronze_customers')}")
versions = spark.sql(f"SELECT customer_id FROM {T('dim_customer')} GROUP BY customer_id HAVING count(*) > 1").count()
assert n(f"SELECT count(*) FROM {T('silver_customers')}") == customers, f"silver_customers needs one row for each of the {customers:,} customers (Task 4)"
assert n(f"SELECT count(*) FROM {T('dim_customer')} WHERE __END_AT IS NULL") == customers, f"dim_customer needs one current row per customer (Task 5)"
assert versions == changed, f"{versions} customers have older versions in dim_customer, Wave 2 changed {changed} (Task 5)"

products = n(f"SELECT count(DISTINCT product_id) FROM {T('bronze_products')}")
rows, ids = spark.sql(f"SELECT count(*), count(DISTINCT product_id) FROM {T('dim_product')}").first()
assert rows == ids == products, f"dim_product needs one row for each of the {products:,} products (Task 6)"

facts = n(f"SELECT count(*) FROM {T('fact_sales')}")
silver = n(f"SELECT count(*) FROM {T('silver_orders')}")
assert facts == silver, f"fact_sales has {facts:,} rows, silver_orders {silver:,} (Task 7)"
no_customer = n(f"SELECT count(*) FROM {T('fact_sales')} WHERE customer_id NOT IN (SELECT customer_id FROM {T('dim_customer')})")
no_product = n(f"SELECT count(*) FROM {T('fact_sales')} WHERE product_id NOT IN (SELECT product_id FROM {T('dim_product')})")
print(f"Run 3 OK: {versions} customers with history, {products:,} products, {facts:,} facts "
      f"({no_customer:,} with an unknown customer, {no_product:,} with an unknown product)")

# History of CUST000001: two versions in dim_customer, one row in silver_customers
display(spark.sql(f"""
    SELECT 'dim_customer (history)' AS table, city, customer_segment, __START_AT, __END_AT
    FROM {T('dim_customer')} WHERE customer_id = 'CUST000001'
    UNION ALL
    SELECT 'silver_customers (current)', city, customer_segment, NULL, NULL
    FROM {T('silver_customers')} WHERE customer_id = 'CUST000001'
    ORDER BY table, __START_AT"""))

## Task 8 (stretch): check

Run this cell after the two full refreshes of `silver_orders` in Task 8: the one that failed, and the one after you put back the rule of Task 2. The first output lists the error events of the failed update: `message` names the flow that failed, for example `Flow '…silver_orders' has FAILED`, and `broken_rule` is the rule that the event quotes after `Violated expectations`. The next line prints that rule, `non_negative_amount`. The pipeline UI shows the rule and the row that broke it in the error list of the update. If the cell fails with `NameError`, see the note under **Build the pipeline**.

In [ ]:
# Task 8: show the failed update and the rule that stopped it, and check that the returns are back in Silver
# broken_rule: the name after "Violated expectations: " in the message or details (JSON can write the quote as u0027)
ERRORS = f"""
    SELECT timestamp,
           regexp_extract(concat_ws(' ', message, details), 'Violated expectations: [^a-z]*(?:u0027)?([a-z][a-z0-9_]*)', 1) AS broken_rule,
           left(message, 300) AS message
    FROM event_log(TABLE({T('silver_orders')}))
    WHERE level = 'ERROR' AND (message ILIKE '%non_negative_amount%' OR details ILIKE '%non_negative_amount%')"""
# the row that names the broken rule first (the UI error list shows the same one)
display(spark.sql(f"{ERRORS} ORDER BY (message ILIKE '%Violated expectations%') DESC, timestamp DESC"))
rules = [r[0] for r in spark.sql(f"SELECT DISTINCT broken_rule FROM ({ERRORS}) WHERE broken_rule <> ''").collect()]
returns_silver = n(f"SELECT count(*) FROM {T('silver_orders')} WHERE total_amount < 0")
assert n(f"SELECT count(*) FROM ({ERRORS})") > 0, "No failed update on non_negative_amount yet: run the full refresh of silver_orders with the code of Task 8"
assert returns_silver > 0, "silver_orders has no returns: put back the rule of Task 2 and run the full refresh of silver_orders again"
if rules:
    print(f"Broken rule (Violated expectations in the event log): {', '.join(rules)}")
else:  # the event names the rule, but not in the form 'Violated expectations: ...'
    print("Broken rule: non_negative_amount (named in the event details; the error list of the update in the pipeline UI shows the row)")
print(f"Task 8 OK: the rule stopped the update; with the rule of Task 2, silver_orders keeps {returns_silver:,} returns")

## Task 9 (stretch): check

Run this cell after **Run file** on `03_gold`. It checks that `daily_sales` adds up to `fact_sales`. The second output comes from the event log and shows how many rows each table wrote in each run. AUTO CDC flows report no row count, so they are not in this list. If the cell fails with `NameError`, see the note under **Build the pipeline**.

In [ ]:
# Task 9: check daily_sales against fact_sales and show the rows written per run
lines = n(f"SELECT sum(order_lines) FROM {T('daily_sales')}")
facts = spark.table(T("fact_sales")).count()
assert lines == facts, f"daily_sales counts {lines:,} order lines, fact_sales has {facts:,} (Task 9)"
print(f"Task 9 OK: daily_sales adds up to the {facts:,} rows of fact_sales")

# Rows written per run and table, oldest run first (AUTO CDC flows report no row count)
display(spark.sql(f"""
    SELECT left(origin.update_id, 8) AS run,
           element_at(split(origin.flow_name, '[.]'), -1) AS flow,
           sum(TRY_CAST(details:flow_progress.metrics.num_output_rows AS BIGINT)) AS rows_written
    FROM event_log(TABLE({T('silver_orders')}))
    WHERE event_type = 'flow_progress' AND origin.flow_name IS NOT NULL
    GROUP BY origin.update_id, origin.flow_name
    HAVING rows_written IS NOT NULL
    ORDER BY min(timestamp), flow"""))

## Clean up (after the course)

Run this only after the course. First delete the pipeline in **Jobs & Pipelines**. After the course the notebook session has usually expired, so run the two Setup cells first (`%run` and **Lab 07 names**). Then select all three lines of the cell below, remove the `#` at the start of each line (Cmd+/ or Ctrl+/) and run it. It drops the schema and the landing folders, so you can start the lab again from Setup.

In [ ]:
# spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{PIPELINE_SCHEMA} CASCADE")
# dbutils.fs.rm(SOURCE_PATH, recurse=True)
# print(f"Dropped the schema {CATALOG}.{PIPELINE_SCHEMA} and removed the landing folders {SOURCE_PATH}")